# Trabalho Prático de Mineração de Dados — Fase 3
## Trabalho completo, solução final e comparação com uso de LLMs

**Disciplina:** Mineração de Dados  
**Grupo:** Grupo 13 - Arthur Faria (2024013915), Felipe Pereira (2024080051), Giordano Liporati (2023027963), João Vaz (2023099166)  
**Dataset:** Spotify Tracks Dataset - https://www.kaggle.com/datasets/yashdev01/spotify-tracks-dataset

**Tarefa de mineração:** Agrupamento  
**Data da entrega:** 31/05/2026



# 0. Controle da entrega e rastreabilidade

| Item | Valor |
|---|---|
| Nome do dataset | Spotify Tracks Dataset |
| Link público do dataset | https://www.kaggle.com/datasets/yashdev01/spotify-tracks-dataset |
| Número de linhas | 92.524 (após limpeza inicial e remoção de duplicatas) |
| Número de colunas | 21 originais |
| Tarefa de mineração | Agrupamento |
| Diretriz 1 de IA responsável | Eficiência e escalabilidade |
| Diretriz 2 de IA responsável | Usabilidade e acessibilidade |
| LLM usada na Fase 2 — baseline | ChatGPT |
| LLM usada na Fase 2 — guiada | ChatGPT  |

## Resumo das decisões herdadas da Fase 2

| Decisão | Origem | Foi mantida na Fase 3? | Justificativa |
|---|---|---|---|
| **Uso do algoritmo MiniBatchKMeans** | Trilha B (Guiada) | Sim | A experimentação prática validou que o agrupamento em lote é essencial para a viabilidade computacional em uma base de mais de 90 mil registros, superando gargalos de algoritmos como o Clustering Hierárquico. |
| **Aplicação de PCA (restando 90% da variância)** | Trilha B (Guiada) | Sim | Reduziu a redundância entre variáveis fortemente correlacionadas e melhorou o tempo de execução. O *trade-off* na interpretabilidade direta dos eixos foi aceito em favor da eficiência. |
| **Manutenção dos Outliers** | Grupo / Baseline | Sim | Valores extremos em `duration_ms` ou `instrumentalness` refletem características musicais legítimas (ex: faixas instrumentais longas). Sua remoção eliminaria nichos importantes do catálogo. |
| **Remoção de Identificadores e Colunas Artificiais** | Grupo | Sim | Colunas como `Unnamed: 0`, `track_id`, `artists` e `album_name` aumentam a cardinalidade sem contribuir para o cálculo de similaridade geométrica acústica, sendo descartadas da modelagem. |
| **Remoção da variável `popularity` do clustering** | Grupo | Sim | A popularidade reflete o sucesso de mercado, e não o áudio em si. Sua inclusão enviesaria os clusters para agrupar "hits" independentemente de serem sonoramente parecidos. |



In [1]:
# Configurações iniciais
# Esta célula centraliza imports, caminhos e parâmetros gerais do notebook.

import os
import time
import json
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

# Caminho para o dataset.
# Substitua pelo caminho real do arquivo usado pelo grupo.
DATA_PATH = "spotify-tracks-dataset.csv"

# Tipo de tarefa do TP.
# Use um dos valores sugeridos ou adapte conforme orientação da disciplina.
# Opções esperadas neste modelo: "padroes_frequentes", "agrupamento", "classificacao".
TASK_TYPE = "agrupamento"

# Coluna-alvo, quando aplicável.
# Para classificação, deve indicar a variável de classe.
# Para clustering ou padrões frequentes, normalmente será None.
TARGET_COLUMN = None  # Exemplo: "classe"

# Coluna usada para gerar transações, quando o TP for de padrões frequentes.
TRANSACTION_COLUMN = None

# Diretório opcional para salvar tabelas, figuras ou artefatos.
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Configuração inicial carregada.")
print(f"TASK_TYPE = {TASK_TYPE}")
print(f"DATA_PATH = {DATA_PATH}")

Configuração inicial carregada.
TASK_TYPE = agrupamento
DATA_PATH = spotify-tracks-dataset.csv


# 1. Business Understanding

## 1.1 Problema de negócio e objetivo analítico

- **Contexto:** Na indústria de streaming, o Spotify lida com um catálogo de milhões de faixas. A taxonomia tradicional baseada em "gêneros" (Rock, Pop) é frequentemente genérica e falha em capturar o perfil acústico das músicas, limitando a experiência do usuário.
- **Problema:** É necessário segmentar de forma objetiva um catálogo musical massivo, agrupando faixas que possuem características sonoras semelhantes (como energia, positividade e ritmo), de maneira escalável e independentemente da popularidade da faixa.
- **Objetivo analítico:** Aplicar algoritmos de Agrupamento sobre as *features* de áudio extraídas da API do Spotify para descobrir perfis acústicos latentes, encontrando um equilíbrio entre a qualidade geométrica dos agrupamentos e o custo computacional.
- **Usuários ou interessados:** Times de Curadoria de Playlists, Engenharia de Recomendação e Marketing do Spotify.
- **Decisões apoiadas:** O agrupamento permitirá automatizar a criação de *playlists* contextuais (ex: "Foco", "Alta Energia"), otimizar as recomendações de faixas similares e analisar lacunas ou tendências no catálogo de áudio.

### Ajustes em relação à Fase 2

O problema e o objetivo centrais foram mantidos, mas a expectativa sobre as fronteiras dos grupos foi refinada após a Fase 2. Observamos empírica e qualitativamente que as propriedades musicais formam um espectro contínuo, resultando em métricas internas moderadas/baixas (como um *Silhouette Score* de 0,09). 

Em vez de buscar separações matemáticas perfeitas (irrealistas para o domínio musical), ajustamos o escopo para focar na **utilidade prática**. Assumimos o *trade-off* de utilizar K=15 no modelo `MiniBatchKMeans` com PCA: abrimos mão da interpretabilidade matemática perfeita e da granularidade exaustiva em prol da agilidade de processamento e de uma segmentação que, via avaliação de centróides, se provou altamente coerente e útil para a equipe de negócios.

## 1.2 Dataset, origem e características finais

### Identificação do dataset

- **Nome:** Spotify Tracks Dataset
- **Fonte:** API do Spotify / Kaggle (por Yash Dev)
- **Link público:** https://www.kaggle.com/datasets/yashdev01/spotify-tracks-dataset
- **Data de acesso ou download:** 17/05/2026

### Características gerais

| Aspecto | Descrição |
|---|---|
| O que cada linha representa? | Uma faixa (música) individual do catálogo, descrita por suas propriedades sonoras. |
| O que cada coluna representa? | Características acústicas objetivas, metadados de identificação e métricas de popularidade. |
| Número de instâncias | 92.524 registros (após limpeza inicial de dados nulos e duplicados). |
| Número de atributos | 21 originais (reduzidos na etapa de modelagem). |
| Tipos principais de atributos | Numéricos contínuos (métricas acústicas), Numéricos discretos (tempo, duração) e Categóricos (nomes, gêneros). |
| Existe variável-alvo? | NÃO. A tarefa é de aprendizado não supervisionado (*clustering*). |
| Existem atributos sensíveis ou identificadores? | SIM. `track_id`, `artists`, `album_name` e `track_name` atuam como identificadores e foram excluídos da modelagem para focar estritamente na similaridade acústica. |

### Dicionário de dados

| Coluna | Tipo | Descrição | Papel na análise | Observações |
|---|---|---|---|---|
| **track_id / track_name** | Texto | Identificador e nome da música. | **Removida** | Útil apenas na etapa final para inspecionar e rotular os clusters. Removida da modelagem. |
| **artists / album_name** | Texto | Nome do artista e álbum. | **Removida** | Alta cardinalidade; retirada para não afetar o cálculo de distância euclidiana. |
| **danceability** | Numérico | O quão adequada a faixa é para dançar. | **Feature** | Escala 0 a 1. Fundamental para capturar o ritmo e estabilidade. |
| **energy** | Numérico | Medida de intensidade e atividade. | **Feature** | Escala 0 a 1. Muito correlacionada com peso e barulho. |
| **acousticness** | Numérico | Grau acústico (sem instrumentos eletrônicos). | **Feature** | Escala 0 a 1. Isola faixas orgânicas de sintetizadas. |
| **instrumentalness** | Numérico | Probabilidade de ausência de vocais. | **Feature** | Escala 0 a 1. Separa faixas instrumentais/lo-fi das cantadas. |
| **valence** | Numérico | A "positividade" musical transmitida. | **Feature** | Escala 0 a 1. Traduz o humor (melancólico vs. eufórico). |
| **speechiness / liveness** | Numérico | Presença de palavras faladas e platéia ao vivo. | **Feature** | Escala 0 a 1. Distingue rap/podcasts e shows ao vivo. |
| **loudness / tempo / duration_ms** | Numérico | Volume (dB), BPM e Duração (ms). | **Feature** | **Requerem padronização rigorosa**, pois as escalas destoam drasticamente do resto. |
| **popularity** | Numérico | Indicador comercial da faixa. | **Removida** | Retirada por carregar viés comercial (não-acústico). |
| **track_genre** | Categórico | Gênero oficial da faixa. | **Removida** | Isolada da modelagem; usada apenas para **validação externa** dos clusters. |
| **Unnamed: 0 / Unnamed: 0.1** | Numérico | Índices de extração. | **Removida** | Resíduos sujos da base original. |

## 1.3 Diretrizes de IA responsável e impacto na solução final

| Diretriz | Pertinência ao problema | Como influenciou a solução final | Evidência concreta |
|---|---|---|---|
| **Eficiência e escalabilidade** | Com mais de 92 mil registros após a limpeza, algoritmos iterativos e baseados em distância (como Hierárquico ou K-Means tradicional sem otimização) sofrem gargalos severos de memória e tempo de execução, inviabilizando a análise rápida. | Alterou a arquitetura do *pipeline*: forçou a implementação de PCA para reduzir a dimensionalidade espacial e determinou a substituição de algoritmos custosos pela abordagem em lotes (`MiniBatchKMeans`). | O algoritmo Hierárquico foi formalmente rejeitado do código final. O uso do `MiniBatchKMeans` com PCA resultou em um tempo de execução otimizado de apenas ~3,32 segundos para todo o dataset. |
| **Usabilidade e acessibilidade** | O modelo gera saídas matemáticas abstratas (ex: "Cluster 3"). Para que o projeto apoie decisões reais (Curadoria de Playlists), os perfis sonoros precisam ser imediatamente compreensíveis para colaboradores não técnicos. | Moldou a preparação (rejeição de codificações hipercomplexas sugeridas pela LLM) e a apresentação final dos resultados, exigindo ferramentas visuais e a tradução das métricas acústicas em personas musicais claras. | Adoção de **Gráficos de Radar** (*Spider Charts*) para visualizar o DNA sonoro de cada centróide, além da rejeição da codificação cíclica (seno/cosseno) da variável `key` para manter a simplicidade analítica. |

### Análise

A explicitação das diretrizes de Inteligência Artificial Responsável transformou o desenvolvimento do projeto, alterando decisões cruciais nas etapas de preparação, modelagem e avaliação. 

As alterações impostas pelas diretrizes foram justificadas do ponto de vista técnico. O impacto mais notável ocorreu na etapa de modelagem devido à diretriz de **Eficiência e Escalabilidade**. Ao lidar com um espaço denso de mais de 90 mil instâncias, calcular a matriz de distâncias de um *Clustering* Hierárquico resultaria em complexidade quadrática $O(n^2)$, estourando a memória RAM. A solução encontrada (PCA + `MiniBatchKMeans`) mitigou ativamente esse risco, garantindo uma execução em pouco mais de 3 segundos.

Contudo, essa responsabilidade técnica gerou **trade-offs** importantes e conscientes assumidos pelo grupo:
1. **Desempenho vs. Interpretabilidade Geométrica:** Para escalar a solução, aplicamos o PCA retendo 90% da variância. O *trade-off* é que os eixos que o `MiniBatchKMeans` utiliza para calcular as distâncias perdem sua interpretabilidade humana direta (deixam de ser *energy* ou *danceability* para se tornarem "Componente 1" e "Componente 2"). 
2. **Qualidade Geométrica vs. Viabilidade Operacional:** O `MiniBatchKMeans` sacrifica uma fração mínima de precisão na convergência dos centróides em troca de ganho massivo de velocidade. 
3. **Completude vs. Acessibilidade:** A diretriz de **Usabilidade** forçou o grupo a estabelecer o hiperparâmetro K=15. Embora números maiores pudessem capturar nuances musicais hiperespecíficas, eles gerariam uma fadiga de análise para o tomador de decisão. Manteve-se a simplicidade analítica em detrimento da completude exaustiva.

Nenhuma das diretrizes se mostrou inaplicável. Pelo contrário, elas atuaram como forças de balanceamento. Enquanto a Eficiência "escondeu" as variáveis originais no PCA durante o treinamento matemático, a Usabilidade exigiu que o grupo resgatasse essas variáveis originais na etapa de avaliação para plotar os Gráficos de Radar, garantindo que o relatório final fosse uma ferramenta acionável, visual e focada no contexto de negócio do Spotify, e não apenas um experimento acadêmico isolado.

# 2. Data Understanding & Data Preparation

Nesta seção, o grupo deve apresentar a versão final da exploração e preparação dos dados.

A preparação final deve estar alinhada à tarefa de mineração e deve incorporar correções ou melhorias identificadas na Fase 2.

A seção deve incluir:

- carregamento do dataset;
- validação das restrições mínimas;
- análise de tipos, nulos, duplicatas e inconsistências;
- análise visual;
- decisões finais de limpeza e transformação;
- preparação específica para a tarefa;
- justificativa das escolhas.

In [ ]:
# Carregamento do dataset
# Adapte esta célula ao formato real: CSV, XLSX, JSON, Parquet etc.

if DATA_PATH == "[INSERIR_CAMINHO_DO_ARQUIVO]":
    print("ATENÇÃO: substitua DATA_PATH pelo caminho real do dataset antes de executar o notebook.")
    df = None
else:
    # Exemplo para CSV. Ajuste separador, encoding e demais parâmetros se necessário.
    df = pd.read_csv(DATA_PATH)
    print("Dataset carregado com sucesso.")
    print(f"Linhas: {df.shape[0]}")
    print(f"Colunas: {df.shape[1]}")
    display(df.head())

In [ ]:
# Validação das restrições mínimas da especificação
# A especificação exige dataset publicamente acessível, com pelo menos 1000 linhas e 4 colunas.
# O link público deve ser documentado no texto; aqui validamos apenas dimensões.

def verificar_restricoes_dataset(df: pd.DataFrame) -> pd.DataFrame:
    if df is None:
        print("Dataset não carregado.")
        return pd.DataFrame()

    n_linhas, n_colunas = df.shape

    validacao = pd.DataFrame([
        {
            "criterio": "Mínimo de 1000 linhas",
            "valor_observado": n_linhas,
            "atende": n_linhas >= 1000
        },
        {
            "criterio": "Mínimo de 4 colunas",
            "valor_observado": n_colunas,
            "atende": n_colunas >= 4
        },
        {
            "criterio": "Dataset publicamente acessível",
            "valor_observado": "[PREENCHER NO TEXTO]",
            "atende": "[PREENCHER]"
        }
    ])

    return validacao

# Exemplo de uso:
# display(verificar_restricoes_dataset(df))

In [ ]:
# Resumo estrutural do dataset

def resumo_estrutural(df: pd.DataFrame) -> pd.DataFrame:
    if df is None:
        print("Dataset não carregado.")
        return pd.DataFrame()

    resumo = pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "n_nulos": df.isna().sum(),
        "perc_nulos": (df.isna().mean() * 100).round(2),
        "n_unicos": df.nunique(dropna=True),
        "exemplo_valor": df.apply(lambda col: col.dropna().iloc[0] if col.dropna().shape[0] > 0 else np.nan)
    })

    return resumo.sort_values(by=["perc_nulos", "n_unicos"], ascending=[False, False])

# Exemplo de uso:
# resumo = resumo_estrutural(df)
# display(resumo)

## 2.1 Interpretação da exploração inicial

[INSERIR INTERPRETAÇÃO DA EXPLORAÇÃO]

Comente:

- quantidade de linhas e colunas;
- tipos de variáveis;
- valores ausentes;
- duplicatas;
- colunas com alta cardinalidade;
- possíveis identificadores;
- possíveis atributos sensíveis;
- atributos relevantes para a tarefa;
- limitações percebidas.

### Relação com decisões da Fase 2

[EXPLICAR QUAIS ALERTAS OU SUGESTÕES DA LLM FORAM CONFIRMADOS OU CORRIGIDOS APÓS A EXPLORAÇÃO REAL]

In [ ]:
# Funções auxiliares para visualização exploratória
# Adapte as colunas e os gráficos ao dataset.

def plot_distribuicao_numerica(df: pd.DataFrame, coluna: str, bins: int = 30) -> None:
    plt.figure(figsize=(8, 4))
    df[coluna].dropna().hist(bins=bins)
    plt.title(f"Distribuição de {coluna}")
    plt.xlabel(coluna)
    plt.ylabel("Frequência")
    plt.tight_layout()
    plt.show()


def plot_contagem_categorica(df: pd.DataFrame, coluna: str, top_n: int = 20) -> None:
    plt.figure(figsize=(10, 4))
    df[coluna].value_counts(dropna=False).head(top_n).plot(kind="bar")
    plt.title(f"Categorias mais frequentes — {coluna}")
    plt.xlabel(coluna)
    plt.ylabel("Frequência")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


def plot_matriz_correlacao(df: pd.DataFrame) -> None:
    numericas = df.select_dtypes(include=np.number)
    if numericas.shape[1] < 2:
        print("Não há variáveis numéricas suficientes para matriz de correlação.")
        return

    corr = numericas.corr()
    plt.figure(figsize=(8, 6))
    plt.imshow(corr, aspect="auto")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
    plt.yticks(range(len(corr.columns)), corr.columns)
    plt.colorbar(label="Correlação")
    plt.title("Matriz de correlação — variáveis numéricas")
    plt.tight_layout()
    plt.show()

# Exemplos de uso:
# plot_distribuicao_numerica(df, "[COLUNA_NUMERICA]")
# plot_contagem_categorica(df, "[COLUNA_CATEGORICA]")
# plot_matriz_correlacao(df)

## 2.2 Análise visual e principais insights

[INSERIR ANÁLISE DOS GRÁFICOS]

A análise deve responder:

- Há distribuições muito assimétricas?
- Existem valores extremos relevantes?
- Existem categorias raras?
- Existe desbalanceamento de classes, se houver variável-alvo?
- Há relações visuais relevantes entre atributos?
- Alguma visualização indica limitação da análise?

> Não invente insights. Toda afirmação deve estar apoiada em gráfico, tabela, estatística descritiva ou inspeção documentada.

In [ ]:
# Verificação simples de duplicatas, nulos e possíveis identificadores

def diagnostico_qualidade(df: pd.DataFrame) -> dict:
    if df is None:
        print("Dataset não carregado.")
        return {}

    diagnostico = {
        "n_linhas": df.shape[0],
        "n_colunas": df.shape[1],
        "n_duplicatas_exatas": int(df.duplicated().sum()),
        "colunas_com_nulos": df.columns[df.isna().any()].tolist(),
        "colunas_possiveis_ids": []
    }

    for col in df.columns:
        proporcao_unicos = df[col].nunique(dropna=True) / max(len(df), 1)
        if proporcao_unicos > 0.95:
            diagnostico["colunas_possiveis_ids"].append(col)

    return diagnostico

# Exemplo de uso:
# diagnostico = diagnostico_qualidade(df)
# print(json.dumps(diagnostico, indent=2, ensure_ascii=False))

## 2.3 Diagnóstico final de qualidade dos dados

| Problema identificado | Evidência | Decisão final | Justificativa |
|---|---|---|---|
| Valores ausentes em [COLUNA] | [INSERIR EVIDÊNCIA] | [REMOVER/IMPUTAR/MANTER] | [JUSTIFICAR] |
| Duplicatas | [INSERIR EVIDÊNCIA] | [DECISÃO] | [JUSTIFICAR] |
| Outliers | [INSERIR EVIDÊNCIA] | [DECISÃO] | [JUSTIFICAR] |
| Alta cardinalidade | [INSERIR EVIDÊNCIA] | [DECISÃO] | [JUSTIFICAR] |
| Atributos identificadores | [INSERIR EVIDÊNCIA] | [DECISÃO] | [JUSTIFICAR] |
| Atributos sensíveis | [INSERIR EVIDÊNCIA] | [DECISÃO] | [JUSTIFICAR] |

### Observação sobre responsabilidade

[EXPLICAR COMO AS DECISÕES DE QUALIDADE DOS DADOS SE RELACIONAM ÀS DIRETRIZES ESCOLHIDAS]

Exemplo: remover um atributo sensível pode reduzir riscos de uso indevido, mas também pode limitar a análise de viés. A decisão precisa ser justificada no contexto do problema.

In [ ]:
# Preparação final dos dados
# Esta função deve ser adaptada pelo grupo.
# Não use transformações automáticas sem justificar no texto.

def preparar_dados_final(df: pd.DataFrame) -> pd.DataFrame:
    if df is None:
        raise ValueError("Dataset não carregado.")

    df_prep = df.copy()

    # 1. Remoção de duplicatas exatas, se justificado.
    df_prep = df_prep.drop_duplicates()

    # 2. Remoção de colunas identificadoras, irrelevantes ou problemáticas.
    # Preencha apenas após justificar na seção textual.
    colunas_remover = [
        # "[COLUNA_IDENTIFICADORA]",
        # "[COLUNA_IRRELEVANTE]",
    ]
    colunas_remover = [col for col in colunas_remover if col in df_prep.columns]
    df_prep = df_prep.drop(columns=colunas_remover)

    # 3. Tratamento simples de nulos em variáveis numéricas.
    # Esta é uma estratégia genérica. Avalie se faz sentido para o dataset.
    for col in df_prep.select_dtypes(include=np.number).columns:
        if df_prep[col].isna().any():
            df_prep[col] = df_prep[col].fillna(df_prep[col].median())

    # 4. Tratamento simples de nulos em variáveis categóricas.
    for col in df_prep.select_dtypes(include=["object", "category"]).columns:
        if df_prep[col].isna().any():
            df_prep[col] = df_prep[col].fillna("desconhecido")

    return df_prep

# Exemplo de uso:
# df_prep = preparar_dados_final(df)
# display(df_prep.head())
# print(df_prep.shape)

## 2.4 Justificativa da preparação final

[INSERIR JUSTIFICATIVA DAS TRANSFORMAÇÕES REALIZADAS]

Explique:

- quais colunas foram removidas;
- como valores ausentes foram tratados;
- se houve normalização, discretização, codificação ou agregação;
- como os dados foram preparados para a tarefa específica;
- quais decisões vieram da baseline;
- quais vieram da trilha guiada;
- quais foram decisões próprias do grupo;
- quais sugestões da LLM foram corrigidas ou descartadas.

In [ ]:
# Preparações específicas por tipo de tarefa
# Use apenas o bloco correspondente ao TASK_TYPE do trabalho.

from typing import List, Tuple, Optional


def gerar_transacoes_de_coluna(df: pd.DataFrame, coluna: str, separador: str = ",") -> List[List[str]]:
    """Gera transações para mineração de padrões frequentes a partir de uma coluna textual."""
    transacoes = []

    for valor in df[coluna].dropna():
        itens = [item.strip().lower() for item in str(valor).split(separador) if item.strip()]
        if itens:
            transacoes.append(itens)

    return transacoes


def preparar_matriz_modelagem(df: pd.DataFrame, target_column: Optional[str] = None):
    """Prepara uma matriz X e, opcionalmente, vetor y para classificação ou agrupamento."""
    df_model = df.copy()

    if target_column is not None:
        y = df_model[target_column]
        X = df_model.drop(columns=[target_column])
    else:
        y = None
        X = df_model

    # Codificação one-hot simples para variáveis categóricas.
    # Em trabalhos finais, justificar se esta codificação é adequada.
    X = pd.get_dummies(X, drop_first=False)

    return X, y

# Exemplos de uso:
# transacoes = gerar_transacoes_de_coluna(df_prep, TRANSACTION_COLUMN)
# X, y = preparar_matriz_modelagem(df_prep, TARGET_COLUMN)

# 3. Modeling

Nesta seção, o grupo deve implementar a solução final.

A modelagem final deve ser resultado de uma decisão crítica, não de simples cópia da LLM.

A seção deve deixar claro:

- quais algoritmos foram testados;
- quais parâmetros foram usados;
- por que esses algoritmos foram escolhidos;
- o que foi inspirado pela baseline;
- o que foi inspirado pela trilha guiada;
- o que foi corrigido ou decidido pelo grupo;
- como a implementação se relaciona às diretrizes de IA responsável.

## 3.1 Comparação entre baseline, guiada e solução final

| Elemento | Sugestão baseline | Sugestão guiada | Solução final do grupo | Justificativa da decisão final |
|---|---|---|---|---|
| Algoritmo(s) | [PREENCHER] | [PREENCHER] | [PREENCHER] | [JUSTIFICAR] |
| Preparação dos dados | [PREENCHER] | [PREENCHER] | [PREENCHER] | [JUSTIFICAR] |
| Parâmetros | [PREENCHER] | [PREENCHER] | [PREENCHER] | [JUSTIFICAR] |
| Métricas | [PREENCHER] | [PREENCHER] | [PREENCHER] | [JUSTIFICAR] |
| Estratégia de avaliação | [PREENCHER] | [PREENCHER] | [PREENCHER] | [JUSTIFICAR] |
| Cuidados de IA responsável | [PREENCHER] | [PREENCHER] | [PREENCHER] | [JUSTIFICAR] |

### Síntese da decisão final

[DESCREVER A SOLUÇÃO FINAL E JUSTIFICAR POR QUE ELA FOI ESCOLHIDA]

A solução final não precisa ser idêntica à melhor resposta da LLM. Ela deve ser tecnicamente defensável e baseada em evidências.

In [ ]:
# Registro simples de experimentos
# Ajuda a manter rastreabilidade de parâmetros e resultados.

experimentos = []


def registrar_experimento(nome: str, algoritmo: str, parametros: dict, metricas: dict, observacoes: str = ""):
    registro = {
        "timestamp": datetime.now().isoformat(timespec="seconds"),
        "nome": nome,
        "algoritmo": algoritmo,
        "parametros": parametros,
        "metricas": metricas,
        "observacoes": observacoes
    }
    experimentos.append(registro)
    return registro


def salvar_experimentos(caminho: str = os.path.join(OUTPUT_DIR, "experimentos.json")):
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(experimentos, f, indent=2, ensure_ascii=False)
    print(f"Experimentos salvos em: {caminho}")

# Exemplo:
# registrar_experimento(
#     nome="teste_inicial",
#     algoritmo="Apriori",
#     parametros={"min_support": 0.05, "min_confidence": 0.6},
#     metricas={"n_regras": 100, "tempo_segundos": 1.23},
#     observacoes="Teste preliminar."
# )

## 3.2 Modelagem para padrões frequentes

Use esta seção se o TP atual for de mineração de padrões frequentes.

Caso o TP seja de agrupamento ou classificação, mantenha apenas como referência e utilize as seções correspondentes abaixo.

In [ ]:
# Modelagem para padrões frequentes: Apriori e FP-Growth
# Esta célula depende de mlxtend. Instale se necessário: pip install mlxtend


def executar_padroes_frequentes(transacoes, min_support=0.05, min_confidence=0.6):
    try:
        from mlxtend.preprocessing import TransactionEncoder
        from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
    except ImportError:
        print("Biblioteca mlxtend não instalada. Instale com: pip install mlxtend")
        return None

    te = TransactionEncoder()
    matriz = te.fit(transacoes).transform(transacoes)
    df_transacoes = pd.DataFrame(matriz, columns=te.columns_)

    resultados = {}

    inicio = time.time()
    itemsets_apriori = apriori(df_transacoes, min_support=min_support, use_colnames=True)
    regras_apriori = association_rules(itemsets_apriori, metric="confidence", min_threshold=min_confidence)
    tempo_apriori = time.time() - inicio

    resultados["Apriori"] = {
        "itemsets": itemsets_apriori,
        "regras": regras_apriori,
        "tempo_segundos": tempo_apriori
    }

    inicio = time.time()
    itemsets_fpgrowth = fpgrowth(df_transacoes, min_support=min_support, use_colnames=True)
    regras_fpgrowth = association_rules(itemsets_fpgrowth, metric="confidence", min_threshold=min_confidence)
    tempo_fpgrowth = time.time() - inicio

    resultados["FP-Growth"] = {
        "itemsets": itemsets_fpgrowth,
        "regras": regras_fpgrowth,
        "tempo_segundos": tempo_fpgrowth
    }

    return resultados


def resumir_padroes_frequentes(resultados):
    if resultados is None:
        return pd.DataFrame()

    linhas = []

    for algoritmo, info in resultados.items():
        itemsets = info["itemsets"]
        regras = info["regras"]

        linhas.append({
            "algoritmo": algoritmo,
            "n_itemsets": len(itemsets),
            "n_regras": len(regras),
            "tempo_segundos": round(info["tempo_segundos"], 4),
            "confianca_media": regras["confidence"].mean() if len(regras) > 0 else np.nan,
            "lift_medio": regras["lift"].mean() if len(regras) > 0 and "lift" in regras.columns else np.nan
        })

    return pd.DataFrame(linhas)

# Exemplo de uso:
# transacoes = gerar_transacoes_de_coluna(df_prep, TRANSACTION_COLUMN)
# resultados_padroes = executar_padroes_frequentes(transacoes, min_support=0.05, min_confidence=0.6)
# resumo_padroes = resumir_padroes_frequentes(resultados_padroes)
# display(resumo_padroes)

## 3.3 Modelagem para agrupamento

Use esta seção se o TP atual for de clusterização.

A avaliação de agrupamento deve considerar métricas internas, interpretação dos grupos e coerência com o problema. Não basta reportar o número de clusters.

In [ ]:
# Modelagem para agrupamento: K-Means e DBSCAN
# Adapte os algoritmos conforme a proposta do grupo.


def executar_agrupamento_basico(X: pd.DataFrame):
    try:
        from sklearn.preprocessing import StandardScaler
        from sklearn.cluster import KMeans, DBSCAN
        from sklearn.metrics import silhouette_score, davies_bouldin_score
    except ImportError:
        print("scikit-learn não está instalado.")
        return pd.DataFrame(), {}

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    resultados = []
    modelos = {}

    # K-Means com alguns valores de k como exemplo.
    for k in [2, 3, 4, 5]:
        inicio = time.time()
        modelo = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = modelo.fit_predict(X_scaled)
        tempo = time.time() - inicio

        if len(set(labels)) > 1:
            sil = silhouette_score(X_scaled, labels)
            db = davies_bouldin_score(X_scaled, labels)
        else:
            sil = np.nan
            db = np.nan

        nome = f"KMeans_k{k}"
        modelos[nome] = modelo
        resultados.append({
            "modelo": nome,
            "n_clusters": len(set(labels)),
            "silhouette": sil,
            "davies_bouldin": db,
            "tempo_segundos": tempo
        })

    # DBSCAN com parâmetros de exemplo.
    inicio = time.time()
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(X_scaled)
    tempo = time.time() - inicio

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters > 1:
        sil = silhouette_score(X_scaled, labels)
        db = davies_bouldin_score(X_scaled, labels)
    else:
        sil = np.nan
        db = np.nan

    modelos["DBSCAN"] = dbscan
    resultados.append({
        "modelo": "DBSCAN",
        "n_clusters": n_clusters,
        "silhouette": sil,
        "davies_bouldin": db,
        "tempo_segundos": tempo
    })

    return pd.DataFrame(resultados), modelos

# Exemplo de uso:
# X, _ = preparar_matriz_modelagem(df_prep, target_column=None)
# resultados_cluster, modelos_cluster = executar_agrupamento_basico(X)
# display(resultados_cluster)

## 3.4 Modelagem para classificação

Use esta seção se o TP atual for de classificação.

A avaliação deve usar uma separação adequada entre treino e teste ou validação cruzada, além de métricas coerentes com o problema, especialmente se houver desbalanceamento de classes.

In [ ]:
# Modelagem para classificação: exemplos com árvore, regressão logística e random forest
# Adapte os modelos conforme a proposta do grupo.


def executar_classificacao_basica(X: pd.DataFrame, y: pd.Series, test_size: float = 0.2):
    try:
        from sklearn.model_selection import train_test_split
        from sklearn.preprocessing import StandardScaler
        from sklearn.pipeline import Pipeline
        from sklearn.tree import DecisionTreeClassifier
        from sklearn.linear_model import LogisticRegression
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    except ImportError:
        print("scikit-learn não está instalado.")
        return pd.DataFrame(), {}

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=y if y.nunique() > 1 else None
    )

    modelos = {
        "DecisionTree": DecisionTreeClassifier(random_state=RANDOM_STATE, max_depth=5),
        "LogisticRegression": Pipeline([
            ("scaler", StandardScaler(with_mean=False)),
            ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
        ]),
        "RandomForest": RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=100)
    }

    resultados = []
    modelos_treinados = {}

    average = "weighted" if y.nunique() > 2 else "binary"

    for nome, modelo in modelos.items():
        inicio = time.time()
        modelo.fit(X_train, y_train)
        y_pred = modelo.predict(X_test)
        tempo = time.time() - inicio

        resultados.append({
            "modelo": nome,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, average=average, zero_division=0),
            "recall": recall_score(y_test, y_pred, average=average, zero_division=0),
            "f1": f1_score(y_test, y_pred, average=average, zero_division=0),
            "tempo_segundos": tempo
        })

        modelos_treinados[nome] = modelo

    return pd.DataFrame(resultados), modelos_treinados

# Exemplo de uso:
# X, y = preparar_matriz_modelagem(df_prep, TARGET_COLUMN)
# resultados_classificacao, modelos_classificacao = executar_classificacao_basica(X, y)
# display(resultados_classificacao)

## 3.5 Resultados da modelagem final

[INSERIR RESULTADOS EFETIVAMENTE OBTIDOS]

Inclua tabelas e comentários sobre:

- algoritmos executados;
- parâmetros usados;
- métricas obtidas;
- tempo de execução;
- estabilidade ou sensibilidade a parâmetros;
- qualidade dos padrões, clusters ou classificações;
- interpretabilidade dos resultados;
- relação com as diretrizes escolhidas.

### Interpretação técnica

[INSERIR INTERPRETAÇÃO]

Evite conclusões vagas. Exemplos de frases fracas:

- “O algoritmo foi bom.”
- “O resultado foi satisfatório.”
- “A LLM ajudou bastante.”

Prefira afirmações verificáveis:

- “O FP-Growth gerou o mesmo conjunto de itemsets que o Apriori, mas com menor tempo de execução neste dataset.”
- “O modelo de árvore teve desempenho inferior, mas permitiu interpretação mais direta das regras de decisão.”
- “O K-Means com k=3 apresentou melhor silhouette entre os valores testados, mas a interpretação dos clusters ainda é limitada.”

# 4. Evaluation

Nesta seção, o grupo deve avaliar a solução final de forma crítica.

A avaliação deve considerar:

- qualidade dos resultados;
- adequação ao problema de negócio;
- desempenho dos algoritmos;
- limitações dos dados;
- limitações da modelagem;
- influência real das LLMs;
- influência real das diretrizes de IA responsável;
- trade-offs observados.

## 4.1 Avaliação dos resultados finais

[INSERIR ANÁLISE DOS RESULTADOS]

A análise deve responder:

- Os resultados são coerentes com o problema?
- Os padrões, clusters ou classificações têm interpretação útil?
- As métricas usadas são adequadas?
- Há sinais de overfitting, instabilidade ou resultado trivial?
- Os resultados dependem fortemente de parâmetros?
- As conclusões são sustentadas pelos dados?

### Limitações dos resultados

[DESCREVER LIMITAÇÕES]

Exemplos:

- dataset desbalanceado;
- atributos pouco informativos;
- ausência de variáveis importantes;
- muitos dados ausentes;
- limitações de escala;
- dificuldade de interpretar padrões numerosos ou redundantes.

## 4.2 Comparação final: baseline × guiada × solução final

| Critério | Baseline sugerida pela LLM | Guiada sugerida pela LLM | Solução final implementada | Avaliação crítica |
|---|---|---|---|---|
| Correção técnica | [ANALISAR] | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Adequação ao problema | [ANALISAR] | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Preparação dos dados | [ANALISAR] | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Algoritmos escolhidos | [ANALISAR] | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Métricas de avaliação | [ANALISAR] | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Interpretabilidade | [ANALISAR] | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Rastreabilidade | [ANALISAR] | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Eficiência computacional | [ANALISAR] | [ANALISAR] | [ANALISAR] | [COMPARAR] |
| Aderência às diretrizes | [ANALISAR] | [ANALISAR] | [ANALISAR] | [COMPARAR] |

### Conclusão da comparação

[INSERIR CONCLUSÃO COMPARATIVA]

A conclusão deve deixar claro:

- o que a baseline acertou;
- o que a baseline ignorou;
- o que a trilha guiada melhorou;
- o que a trilha guiada ainda errou;
- o que o grupo decidiu implementar;
- por que a solução final é tecnicamente mais defensável.

## 4.3 Sugestões aceitas, rejeitadas e corrigidas

| Sugestão | Origem | Decisão | Justificativa |
|---|---|---|---|
| [SUGESTÃO 1] | Baseline | Aceita / Rejeitada / Corrigida | [JUSTIFICAR] |
| [SUGESTÃO 2] | Guiada | Aceita / Rejeitada / Corrigida | [JUSTIFICAR] |
| [SUGESTÃO 3] | Grupo | Implementada | [JUSTIFICAR] |

### Erros ou limitações das LLMs

| Erro, omissão ou limitação | Trilha | Impacto potencial | Correção feita pelo grupo |
|---|---|---|---|
| [DESCREVER] | Baseline / Guiada | [DESCREVER] | [DESCREVER] |
| [DESCREVER] | Baseline / Guiada | [DESCREVER] | [DESCREVER] |

> O grupo deve demonstrar que avaliou criticamente a LLM. Uma boa entrega não aceita automaticamente todas as recomendações geradas.

## 4.4 Avaliação das diretrizes de IA responsável na solução final

| Diretriz | Como apareceu na Fase 2 | Como foi incorporada na Fase 3 | Evidência no notebook | Limitações |
|---|---|---|---|---|
| [DIRETRIZ 1] | [DESCREVER] | [DESCREVER] | [INSERIR EVIDÊNCIA] | [DESCREVER] |
| [DIRETRIZ 2] | [DESCREVER] | [DESCREVER] | [INSERIR EVIDÊNCIA] | [DESCREVER] |

### Discussão crítica

[INSERIR DISCUSSÃO]

Questões orientadoras:

- A diretriz influenciou apenas o texto ou também o pipeline técnico?
- Houve mudança em atributos, algoritmos, métricas ou interpretação?
- A diretriz melhorou a qualidade da solução?
- Houve custo adicional, perda de desempenho ou aumento de complexidade?
- Alguma diretriz foi difícil de operacionalizar? Por quê?

## 4.5 Trade-offs finais

| Trade-off | Evidência observada | Decisão final | Justificativa |
|---|---|---|---|
| Desempenho × interpretabilidade | [PREENCHER] | [PREENCHER] | [JUSTIFICAR] |
| Privacidade × utilidade dos dados | [PREENCHER] | [PREENCHER] | [JUSTIFICAR] |
| Eficiência × qualidade dos resultados | [PREENCHER] | [PREENCHER] | [JUSTIFICAR] |
| Simplicidade × completude da análise | [PREENCHER] | [PREENCHER] | [JUSTIFICAR] |
| Justiça/viés × disponibilidade de atributos | [PREENCHER] | [PREENCHER] | [JUSTIFICAR] |

### Síntese

[INSERIR SÍNTESE DOS TRADE-OFFS MAIS IMPORTANTES]

Nem todos os trade-offs serão aplicáveis a todos os trabalhos. Discuta apenas os que fizerem sentido para o dataset e para a tarefa.

# 5. Conclusão

[INSERIR CONCLUSÃO FINAL]

A conclusão deve retomar:

1. o objetivo do trabalho;
2. a solução final implementada;
3. os principais resultados obtidos;
4. as limitações da análise;
5. o papel das LLMs;
6. o efeito das diretrizes de IA responsável;
7. possíveis melhorias futuras.

## Exemplo de estrutura de conclusão

> Neste trabalho, investigamos [PROBLEMA] usando o dataset [DATASET] e aplicamos [TÉCNICA]. Após comparar as sugestões da LLM nas condições baseline e guiada, decidimos implementar [SOLUÇÃO FINAL]. Os resultados indicam que [SÍNTESE DOS RESULTADOS], com limitações relacionadas a [LIMITAÇÕES]. A explicitação das diretrizes de [DIRETRIZ 1] e [DIRETRIZ 2] contribuiu para [EFEITO CONCRETO], embora [TRADE-OFF OU LIMITAÇÃO].

> Substitua o exemplo por uma conclusão real, baseada nos resultados executados.

# 6. Checklist final da Fase 3

Antes de entregar, verifique se o notebook contém:

- [ ] identificação do grupo;
- [ ] link público do dataset;
- [ ] validação das restrições mínimas do dataset;
- [ ] Business Understanding consolidado;
- [ ] descrição clara do dataset;
- [ ] dicionário de dados;
- [ ] diretrizes de IA responsável escolhidas;
- [ ] explicação de como as diretrizes influenciaram a solução final;
- [ ] Data Understanding com tabelas, gráficos e interpretação;
- [ ] Data Preparation com justificativa das decisões;
- [ ] modelagem final executada;
- [ ] parâmetros documentados;
- [ ] métricas adequadas à tarefa;
- [ ] resultados interpretados;
- [ ] comparação baseline × guiada × solução final;
- [ ] sugestões aceitas, rejeitadas e corrigidas;
- [ ] erros e limitações das LLMs;
- [ ] trade-offs discutidos;
- [ ] conclusão final baseada em evidências;
- [ ] referências utilizadas.

# 7. Referências

[INSERIR REFERÊNCIAS UTILIZADAS]

Inclua, quando aplicável:

- documentação do dataset;
- materiais da disciplina;
- documentação das bibliotecas;
- referências sobre CRISP-DM;
- referências sobre algoritmos utilizados;
- referências sobre IA responsável;
- artigos ou livros usados para embasar decisões técnicas.

> As referências devem apoiar decisões reais do trabalho. Evite listar referências que não foram usadas na análise.

# 8. Apêndice opcional — detalhes adicionais

Use esta seção para incluir materiais complementares que não cabem no corpo principal do notebook, por exemplo:

- tabelas extensas;
- resultados de experimentos secundários;
- variações de parâmetros;
- prompts resumidos da Fase 2, se o grupo julgar necessário;
- detalhes de implementação;
- análises adicionais não centrais.

> O apêndice não substitui as seções principais. Ele deve apenas complementar a argumentação.

In [ ]:
# Célula opcional para salvar uma versão resumida dos principais artefatos gerados
# Adapte conforme os objetos produzidos no notebook.

# Exemplos:
# resumo_padroes.to_csv(os.path.join(OUTPUT_DIR, "resumo_padroes.csv"), index=False)
# resultados_cluster.to_csv(os.path.join(OUTPUT_DIR, "resultados_cluster.csv"), index=False)
# resultados_classificacao.to_csv(os.path.join(OUTPUT_DIR, "resultados_classificacao.csv"), index=False)
# salvar_experimentos()

print("Use esta célula apenas após gerar os artefatos finais do trabalho.")